# 01 - Reproduction Experiments

This notebook is the main reproduction notebook for the project.

Scope:
- Table 1: main comparison on CIFAR-10, CIFAR-100-20, STL-10, ImageNet-10
- Table 2: convergence of inferred K
- Table 3: ImageNet-50 paper summary and local status
- Deviation analysis for the reproduction section


## Optional Regeneration

Current local PnP preset in this repo:
- `epochs=80`
- `warmup_epochs=30`
- `lambda_param=2.0`

The next code cell refreshes the structured Table 1 view whenever the source CSV is newer so the notebook does not keep stale numbers.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path('..').resolve()
RESULT_DIR = ROOT / 'data' / 'scan_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

required_tables = [
    RESULT_DIR / 'table1_main_comparison.csv',
    RESULT_DIR / 'table1_paper_vs_local.csv',
    RESULT_DIR / 'table2_inferred_k.csv',
    RESULT_DIR / 'table3_imagenet50_summary.csv',
]

if any(not path.exists() for path in required_tables):
    subprocess.run(
        ['python', 'scripts/generate_reproduction_tables.py', '--tables', 'table1', 'table2', 'table3'],
        check=True,
        cwd=ROOT,
    )

main_table_path = RESULT_DIR / 'table1_main_comparison.csv'
structured_path = RESULT_DIR / 'table1_structured_for_report.csv'
if (not structured_path.exists()) or structured_path.stat().st_mtime < main_table_path.stat().st_mtime:
    subprocess.run(['python', 'scripts/format_table1.py'], check=True, cwd=ROOT)

table1 = pd.read_csv(structured_path)
comparison_table = pd.read_csv(RESULT_DIR / 'table1_paper_vs_local.csv')
table2 = pd.read_csv(RESULT_DIR / 'table2_inferred_k.csv')
table3 = pd.read_csv(RESULT_DIR / 'table3_imagenet50_summary.csv')

RESULT_DIR


## Table 1

Structured view for the main comparison section of the report. This table is now synchronized with `table1_main_comparison.csv`.


In [ ]:
cols = table1.columns.tolist()
multi = []
for c in cols:
    if c == 'Method':
        multi.append(('Method', 'Metric'))
    else:
        ds, metric = c.split('|')
        multi.append((ds, metric))

table1_multi = table1.copy()
table1_multi.columns = pd.MultiIndex.from_tuples(multi)
display(table1_multi)


## Deviation Analysis

The reproduction requirement asks for explicit paper-vs-local comparison together with relative deviation. The next cells expose that comparison directly instead of relying only on the formatted report table.


In [ ]:
mean_deviation = (
    comparison_table.groupby(['Dataset', 'Paper method'])['Relative deviation(%)']
    .mean()
    .reset_index()
    .sort_values('Relative deviation(%)', ascending=False)
)
display(mean_deviation)

worst_rows = comparison_table.sort_values('Relative deviation(%)', ascending=False).head(12)
display(worst_rows)


## Interpretation

Main takeaways for the report text:
- `SCAN (local)` is reasonably close to the paper on `cifar-20` and `stl-10`, and even exceeds the paper on `imagenet-10`.
- The largest gaps appear in `Ours (K0=3)`, especially on `stl-10` and `imagenet-10`. This suggests the local split and merge schedule is still sensitive when the initial cluster count starts far below the target.
- The `K0=20` or `K0=30` runs are much closer to the paper, so the current implementation is better at shrinking toward the target than growing from a strongly under-clustered start.
- Because the backbone features are frozen and imported from upstream SCAN or MoCo checkpoints, the remaining mismatch is most likely concentrated in head initialization, split and merge timing, and threshold dynamics rather than feature extraction itself.


## Table 2

Inferred K convergence compared with the paper.


In [ ]:
table2 = table2.sort_values(['Dataset', 'K0']).reset_index(drop=True)
display(table2)


## Table 3

ImageNet-50 remains separated here as a paper-summary section because the dataset is not available in the current workspace.


In [ ]:
display(table3)
